# 01 · RQ1 lifecycle

**Question.** How do trends rise, peak and fade, and how long do they last? The half-life sets the days horizon's baseline ("will it still matter when my video is ready?"), and the shape of the rise says how early a trend can be caught.

**Data.** The stored lists (`trend_items`), per platform, each list's top 10. Once topic snapshots exist (after phase 2), repeat this on them: a topic's life across platforms and in the combined top 10, by category (from clustering the topic centroids).

**Method.** A *spell* is one stay in a list's top 10: from the fetch it entered to the first fetch it's missing from. Items are matched across fetches by url for Bluesky and Hacker News, by title elsewhere.
- *left-censored*: already listed at the list's first fetch, or right after a gap in its fetches. The start is unknown, so it's left out;
- *right-censored*: still listed at the end of the data, or before a gap. It hasn't ended, and the survival curves (Kaplan–Meier, `lifelines`) count it as ongoing;
- `max_missed=1` lets a stay carry on through one missed fetch. Bluesky's list drops and re-adds topics from hour to hour (most returns come after one missed fetch).

Fetches are hourly, so durations are good to about an hour. TikTok, Instagram and Pinterest refresh daily or less often and are left out. Google's feed is the 10 newest trends (RQ5), so its time in the list is the time until 10 newer ones start, not its lifespan.

Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from lifelines import KaplanMeierFitter

from topnews import db, lifecycle, rhythms

engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
items = db.trend_items(engine, days=28)  # YouTube left out
fast = items[~items.source_id.isin(rhythms.SLOW_SOURCES)]
rhythms.coverage(fast)

## Spells and censoring

In [ ]:
strict = lifecycle.spells(fast)
tolerant = lifecycle.spells(fast, max_missed=1)
strict.groupby("list_id").agg(
    spells=("key", "size"), left_censored=("left_censored", "sum"), right_censored=("right_censored", "sum")
)

## How long trends stay in the top 10

`median_hours` is the half-life: half the trends that entered were gone by then. `inf` means fewer than half have left yet, so the data is too short to say. `listed_after_Nh` is the share still in the top 10 N hours after entering.

In [ ]:
pd.concat(
    {"strict": lifecycle.survival(strict).set_index("list_id"), "one missed fetch allowed": lifecycle.survival(tolerant).set_index("list_id")},
    axis=1,
).round(2).T

In [ ]:
usable = strict[~strict.left_censored]
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, group in usable.groupby("list_id"):
    if len(group) >= 5:
        KaplanMeierFitter().fit(group.hours, ~group.right_censored, label=f"{name} ({len(group)})").plot_survival_function(ax=ax, ci_show=False)
ax.set_xlabel("hours since entering the top 10")
ax.set_ylabel("share still listed")
ax.set_title("Time in the top 10 (Kaplan–Meier)", fontsize=10)
plt.show()

## The shape of a stay

Where trends enter, how often they climb after entering, and how often an item comes back for another stay. Then the median rank by hours since entering, for the trends still listed at that age.

In [ ]:
lifecycle.shapes(strict).round(2)

In [ ]:
paths = lifecycle.rank_paths(fast, strict)
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, group in paths[paths.still_listed >= 3].groupby("list_id"):
    ax.plot(group.age, group.median_rank, marker="o", ms=3, label=name)
ax.invert_yaxis()
ax.set_xlabel("hours since entering the top 10")
ax.set_ylabel("median rank")
ax.set_title("Rank by age, where at least 3 trends are still listed", fontsize=10)
ax.legend(fontsize=8)
plt.show()
paths.pivot(index="age", columns="list_id", values="still_listed")

## The longest finished stays, for reading

In [ ]:
done = strict[~strict.left_censored & ~strict.right_censored]
done.sort_values("hours", ascending=False).groupby("list_id").head(3)[
    ["list_id", "title", "hours", "entry_rank", "best_rank", "exit_rank"]
].round(1)